# Text Mining - Introdução

In [4]:
!pip install nltk
!pip install spacy
!pip install transformers

In [5]:
import pandas as pd
import re
import nltk
import spacy
import string
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
## Ensure you have at least this data:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')
nltk.download('punkt_tab')

#You can install everything available by running this line
#nltk.download()

[nltk_data] Downloading package stopwords to /home/thiago/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/thiago/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/thiago/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/thiago/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Basic Pre-Processing with NLTK

#### Sentence separation

In [7]:
from nltk.tokenize import sent_tokenize

texto = "A fazenda é muito grande. O Dr. da Fazenda possui vários animais. Aqui temos um olho d'água."

sentencas = sent_tokenize(texto)

for sent in sentencas:
    print(sent)

A fazenda é muito grande.
O Dr. da Fazenda possui vários animais.
Aqui temos um olho d'água.


#### Tokenization

In [8]:
from nltk.tokenize import word_tokenize

texto = "A fazenda é muito grande. O Dr. da Fazenda possui vários animais. Aqui temos um olho d'água."

palavras = word_tokenize(texto) 

for palavra in palavras:
    print(palavra)

A
fazenda
é
muito
grande
.
O
Dr.
da
Fazenda
possui
vários
animais
.
Aqui
temos
um
olho
d'água
.


### Illustrating with Yelp review data

In [9]:
dfCharlotte= pd.read_csv("reviewsTestCharlotte.csv")

reviewsCharlotte_original = dfCharlotte['text']
reviewsCharlotte_original

reviewsCharlotte_original[0]

'I loved the customer service! Fast, but the doctor took her time to make sure I was happy in my contact lenses. Thank you!'

In [10]:
reviewsCharlotte_original.shape

(48617,)

#### Converting to lowercase

In [11]:
reviewsCharlotte_prep = reviewsCharlotte_original.str.lower()
reviewsCharlotte_prep

0        i loved the customer service! fast, but the do...
1        i've been coming here for a couple of years bu...
2        i really liked the optometrist here. the wait ...
3        robert and his team took great care of me when...
4        edit: i'm lowering this review from 5-stars to...
                               ...                        
48612    we've now had mike out twice to fix some leaks...
48613    love the new location lauren is always right o...
48614    i completely ruined my hair today attempting t...
48615    i have been going to lauren for years now and ...
48616    always available when i need her, lauren rocks...
Name: text, Length: 48617, dtype: object

#### Remove punctuation

In [12]:
PONTUACAO = string.punctuation

def remove_pontuacao(text):
    return text.translate(str.maketrans('', '', PONTUACAO))

reviewsCharlotte_prep = reviewsCharlotte_prep.apply(lambda text: remove_pontuacao(text))
reviewsCharlotte_prep

0        i loved the customer service fast but the doct...
1        ive been coming here for a couple of years but...
2        i really liked the optometrist here the wait w...
3        robert and his team took great care of me when...
4        edit im lowering this review from 5stars to 3s...
                               ...                        
48612    weve now had mike out twice to fix some leaks ...
48613    love the new location lauren is always right o...
48614    i completely ruined my hair today attempting t...
48615    i have been going to lauren for years now and ...
48616    always available when i need her lauren rocks ...
Name: text, Length: 48617, dtype: object

#### Remove stopwords

In [13]:
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPWORDS])

reviewsCharlotte_prep = reviewsCharlotte_prep.apply(lambda text: remove_stopwords(text))
print(reviewsCharlotte_prep)

print(STOPWORDS)


0        loved customer service fast doctor took time m...
1        ive coming couple years optometrist business b...
2        really liked optometrist wait little longer wo...
3        robert team took great care repairing car back...
4        edit im lowering review 5stars 3stars wrote or...
                               ...                        
48612    weve mike twice fix leaks pex pipes first time...
48613    love new location lauren always right point lo...
48614    completely ruined hair today attempting highli...
48615    going lauren years would never see another hai...
48616    always available need lauren rocks particular ...
Name: text, Length: 48617, dtype: object
{"needn't", "you'd", 'wasn', 'was', 'but', 'those', 'herself', 'yourselves', 'they', "isn't", "she'd", "it'll", "you're", 'them', "shouldn't", 'again', 'against', 'at', 'because', 'below', 'him', 'same', 'hers', "mustn't", 'will', 't', 'while', "wouldn't", "they're", 'had', 'through', 'ain', 's', 'won', 'its'

#### Stemming

In [14]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()

def stem_words(text):
    return " ".join([stemmer.stem(word) for word in text.split()])

reviewsCharlotte_prep= reviewsCharlotte_prep.apply(lambda text: stem_words(text))
reviewsCharlotte_prep

0        love custom servic fast doctor took time make ...
1        ive come coupl year optometrist busi busi yelp...
2        realli like optometrist wait littl longer woul...
3        robert team took great care repair car back ca...
4        edit im lower review 5star 3star wrote origin ...
                               ...                        
48612    weve mike twice fix leak pex pipe first time l...
48613    love new locat lauren alway right point love h...
48614    complet ruin hair today attempt highlight than...
48615    go lauren year would never see anoth hairdress...
48616    alway avail need lauren rock particular red ha...
Name: text, Length: 48617, dtype: object

# Topic Modeling

In [15]:
from gensim.models.ldamodel import LdaModel
from gensim.corpora.dictionary import Dictionary

### No preprocessing

In [16]:
dataset_orig = [d.split() for d in reviewsCharlotte_original]
dictionary_orig = Dictionary(dataset_orig)
corpus_orig = [dictionary_orig.doc2bow(doc) for doc in dataset_orig]

model_original =LdaModel(corpus=corpus_orig, id2word=dictionary_orig, num_topics=10, iterations=100, passes=5,random_state=1)


In [17]:
for i in  model_original.show_topics(num_topics=10, num_words=6, log=False):
    print(i)
    print('---')


(0, '0.023*"games" + 0.010*"cookies" + 0.009*"Asked" + 0.005*"Rooms" + 0.005*"delivered." + 0.004*"purpose"')
---
(1, '0.040*"I" + 0.036*"to" + 0.033*"the" + 0.020*"a" + 0.017*"you" + 0.016*"that"')
---
(2, '0.021*"massage" + 0.008*"Taste" + 0.006*"sake" + 0.006*"Still," + 0.005*"Suites" + 0.005*"Avoid"')
---
(3, '0.005*"Ice" + 0.004*"mold" + 0.004*"Awful" + 0.003*"pleasure." + 0.003*"this?" + 0.003*"Cesar"')
---
(4, '0.050*"the" + 0.040*"and" + 0.039*"was" + 0.030*"I" + 0.026*"a" + 0.016*"The"')
---
(5, '0.015*"Thai" + 0.015*"ribs" + 0.008*"tire" + 0.007*"pork," + 0.007*"Taco" + 0.006*"Fantastic"')
---
(6, '0.012*"Hall" + 0.010*"Western" + 0.004*"tipped" + 0.004*"Yeah," + 0.003*"watch." + 0.002*":)."')
---
(7, '0.045*"hair" + 0.020*"salon" + 0.011*"cut" + 0.010*"color" + 0.010*"polish" + 0.008*"pedicure"')
---
(8, '0.046*"the" + 0.043*"and" + 0.039*"a" + 0.029*"is" + 0.023*"to" + 0.021*"of"')
---
(9, '0.046*"the" + 0.044*"and" + 0.041*"to" + 0.040*"I" + 0.038*"was" + 0.022*"a"')
---


### With preprocessing

In [18]:
dataset_prep = [d.split() for d in reviewsCharlotte_prep]
dictionary_prep = Dictionary(dataset_prep)
corpus_prep = [dictionary_prep.doc2bow(doc) for doc in dataset_prep]

model_prep =LdaModel(corpus=corpus_prep, id2word=dictionary_prep, num_topics=10, iterations=100, passes=5,random_state=1)


In [19]:
for i in  model_prep.show_topics(num_topics=10, num_words=6, log=False):
    print(i)
    print('---')


(0, '0.024*"beer" + 0.014*"bar" + 0.013*"place" + 0.013*"great" + 0.012*"good" + 0.010*"nice"')
---
(1, '0.023*"order" + 0.023*"food" + 0.015*"time" + 0.015*"wait" + 0.014*"us" + 0.013*"minut"')
---
(2, '0.021*"breakfast" + 0.018*"coffe" + 0.017*"egg" + 0.017*"brunch" + 0.014*"ice" + 0.013*"cream"')
---
(3, '0.044*"great" + 0.035*"food" + 0.031*"place" + 0.025*"servic" + 0.019*"good" + 0.018*"love"')
---
(4, '0.022*"call" + 0.016*"would" + 0.012*"told" + 0.011*"custom" + 0.011*"said" + 0.010*"servic"')
---
(5, '0.043*"sub" + 0.027*"gift" + 0.023*"japanes" + 0.022*"groceri" + 0.021*"jerk" + 0.018*"fuel"')
---
(6, '0.020*"good" + 0.019*"burger" + 0.015*"chicken" + 0.014*"fri" + 0.014*"order" + 0.011*"like"')
---
(7, '0.016*"like" + 0.014*"get" + 0.012*"go" + 0.010*"im" + 0.010*"look" + 0.010*"place"')
---
(8, '0.123*"airport" + 0.083*"room" + 0.072*"hotel" + 0.053*"stay" + 0.050*"taco" + 0.030*"flight"')
---
(9, '0.015*"car" + 0.014*"help" + 0.011*"work" + 0.011*"store" + 0.010*"need" + 

### Probabilities of topics in each document

In [20]:
docsProbabilities = model_prep.get_document_topics(corpus_prep, minimum_probability=0)

docsProbabilities

In [21]:
#Probability of each topic per document. Tuple (topic, probability)
for docDistribution in docsProbabilities[0:10]:
    print(docDistribution)

[(0, 0.0071519343), (1, 0.15872245), (2, 0.007151945), (3, 0.25934404), (4, 0.00715396), (5, 0.0071514593), (6, 0.007151999), (7, 0.007152517), (8, 0.0071514593), (9, 0.5318682)]
[(0, 0.0016159642), (1, 0.0016160049), (2, 0.0016156161), (3, 0.001615824), (4, 0.20321089), (5, 0.0016154525), (6, 0.1409837), (7, 0.001615989), (8, 0.0016155083), (9, 0.64449507)]
[(0, 0.001852898), (1, 0.00185335), (2, 0.0018526925), (3, 0.0018531331), (4, 0.0018529615), (5, 0.0018524572), (6, 0.0018528634), (7, 0.5231479), (8, 0.0018524554), (9, 0.46202928)]
[(0, 0.00294224), (1, 0.0029428527), (2, 0.0029422138), (3, 0.0029421574), (4, 0.11584241), (5, 0.0029418338), (6, 0.0029423486), (7, 0.0029422585), (8, 0.0029418422), (9, 0.86061984)]
[(0, 0.08094109), (1, 0.00048701806), (2, 0.00048694856), (3, 0.0004870454), (4, 0.5226361), (5, 0.005358452), (6, 0.029970376), (7, 0.00048702172), (8, 0.00048686858), (9, 0.35865906)]
[(0, 0.33742812), (1, 0.007703148), (2, 0.0077025583), (3, 0.31229982), (4, 0.0077042

## Bertopic

In [22]:
!pip install bertopic

In [23]:
from bertopic import BERTopic

topic_model = BERTopic()

dataset_orig_list = [d for d in reviewsCharlotte_original]
dataset_orig_list

topics, probs = topic_model.fit_transform(dataset_orig_list)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [24]:
#Topics labeled as -1 are considered outliers
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,20427,-1_was_the_and_we,"[was, the, and, we, it, food, were, to, had, but]",[Just okay. My fiancé and I are new to town so...
1,0,1177,0_burger_fries_burgers_potato,"[burger, fries, burgers, potato, pickles, bun,...","[Their sign boasting ""Charlotte's Best Burgers..."
2,1,883,1_bbq_pork_brisket_ribs,"[bbq, pork, brisket, ribs, pulled, barbecue, m...",[I went to lunch at Bill Spoon's Barbecue on a...
3,2,581,2_coffee_starbucks_latte_mugs,"[coffee, starbucks, latte, mugs, shop, wifi, b...",[The best Starbucks in town. I love it here. B...
4,3,570,3_brunch_buffet_sunday_toast,"[brunch, buffet, sunday, toast, bistro, eggs, ...",[This place's brunch is amazing.\n\nWe went fo...
...,...,...,...,...,...
417,416,10,416_zoom_obedience_agility_dog,"[zoom, obedience, agility, dog, training, jule...",[update-I decided to give zoom room another ch...
418,417,10,417_latin_honduran_english_authentic,"[latin, honduran, english, authentic, salvador...",[Best latin food ever and really cheap. It's a...
419,418,10,418_bento_sushi_bentos_buffet,"[bento, sushi, bentos, buffet, rice, tofu, unc...",[Bento is definitely a unique place from what ...
420,419,10,419_blaze_pizza_lemonade_toppings,"[blaze, pizza, lemonade, toppings, preservativ...",[Of all the chain pizzerias I have to go with ...


##### Fine tuning

In [25]:
from bertopic.representation import KeyBERTInspired

representation_model = KeyBERTInspired()
topic_model_fine = BERTopic(representation_model=representation_model)

topics, probs = topic_model_fine.fit_transform(dataset_orig_list)

In [26]:
#Topics labeled as -1 are considered outliers
topic_model_fine.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,20143,-1_restaurant_pizza_food_delicious,"[restaurant, pizza, food, delicious, chicken, ...",[Our night out at Jade with friends for a holi...
1,0,909,0_barbecue_bbq_brisket_ribs,"[barbecue, bbq, brisket, ribs, barbeque, pork,...",[Came into town for a work related conference ...
2,1,834,1_burger_burgers_cheeseburger_fries,"[burger, burgers, cheeseburger, fries, patties...",[Smashburger was a nice little surprise. Havi...
3,2,822,2_hotel_hotels_marriott_rooms,"[hotel, hotels, marriott, rooms, hilton, inn, ...",[TRAVELERS BEWARE! If 0 stars was an option fo...
4,3,594,3_brunch_buffet_restaurant_dishes,"[brunch, buffet, restaurant, dishes, dinner, b...",[So after reading many stellar reviews about t...
...,...,...,...,...,...
399,398,10,398_shirts_shirt_tshirts_tshirt,"[shirts, shirt, tshirts, tshirt, clothes, ward...","[Great design help, pricing was very reasonabl..."
400,399,10,399_midtown_place_breezeway_town,"[midtown, place, breezeway, town, tavern, geor...",[I love this place they are so cool and they a...
401,400,10,400_oysters_oyster_charlotte_seafood,"[oysters, oyster, charlotte, seafood, coast, s...",[This place has some of the freshest seafood i...
402,401,10,401_bentos_sushi_bento_buffet,"[bentos, sushi, bento, buffet, entrees, meals,...","[So, we thought that this place would excel in..."


# Sentiment Analysis

### Using VADER

In [27]:
from nltk.sentiment import SentimentIntensityAnalyzer

sent = SentimentIntensityAnalyzer()

print(sent.polarity_scores("Yes! Data mining is very powerful!"))

{'neg': 0.0, 'neu': 0.385, 'pos': 0.615, 'compound': 0.7489}


The compound score is calculated in such a way that it represents the polarity of -1 to 1, where +1 is the most positive and -1 is the most negative.

### Using Textblob

In [28]:
!pip install -U textblob

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
from textblob import TextBlob

frase = TextBlob("Yes! Data mining is very powerful!")
frase.sentiment
#range -1 to 1

Sentiment(polarity=0.48750000000000004, subjectivity=1.0)

##### With transformers

In [ ]:
from transformers import pipeline

# Load a sentiment analysis pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Analyze text
result = sentiment_pipeline("Yes! Data mining is very powerful!")
print(result)

#probabilidade de estar associado a uma classe: 0.99 = elevada confiança


[{'label': 'POSITIVE', 'score': 0.9998205304145813}]


### Example with Yelp reviews

In [30]:
def verifica_MuitoPositivo(review):
    
    return sent.polarity_scores(review)["compound"] > 0.8


for review in reviewsCharlotte_original[0:10]:
    
    if verifica_MuitoPositivo(review):
        print(review)
        print('----')

I loved the customer service! Fast, but the doctor took her time to make sure I was happy in my contact lenses. Thank you!
----
Robert and his team took great care of me when repairing my car back up camera. The team was able to identify the issue and resolve it for a low cost option. The team could have taken advantage of me and oversold equipment that wasn't needed, however they were very upfront about the root cause of the issue and I was very pleased with the repair and the repair cost !
----
Edit: I'm lowering this review from 5-stars to 3-stars.  I wrote the original review prematurely. After having been around inside my car a bit I have noticed broken rivets & clips and pieces of carpet were never put back where they should have been. Also, many leftover pieces of wire and debris were left in the crevices and sides of the seat. My biggest issue is every time I turn my car on I hear a pop come from the JL subwoofer that was bought and installed at Freeman's. I brought this up to 

# Transformers - Similar phrases with sBert

In [31]:
#!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer, util

model_RobLarge = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')


In [32]:
dfTweetsTrump = pd.read_csv("tweets_trump.csv")

#Using only with the first 10000 -
embeddingsTweets_ROBbase = model_RobLarge.encode(dfTweetsTrump.loc[0:10000,'text'], convert_to_tensor=True)

In [33]:
embeddingsTweets_ROBbase
matSimi = cosine_similarity(embeddingsTweets_ROBbase.cpu().detach().numpy())

In [34]:
maior = 0
count = 0
docID = -1
docTarget = 120

for i in matSimi[docTarget,:]:
    if count == docTarget:
        count +=1
        continue
        
    if i> maior:
        maior = i
        docID = count
    
    count+=1

In [35]:
print('****target***')
print(dfTweetsTrump.loc[docTarget,'text'])
print()
print("*** mais similar ***")
print(dfTweetsTrump.loc[docID,'text'])

print(maior)
print(docID)

****target***
RT @Claire_FOX5: #BREAKING:  @GaSecofState's office confirms Floyd County has found 2,600 ballots during audit.  Says Sec. Raffensperger wa…

*** mais similar ***
Investigators Dispatched After Fulton County Discovers ‘Issue‘ with Ballot Reporting https://t.co/lShmKksQ0O via @BreitbartNews
0.6110223
2582


## Other examples with Transformers

### Classification - zero-shot

In [36]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model='valhalla/distilbart-mnli-12-1')


In [37]:
classifier(
    ["This is the Data Mining course from CPGEI",
     "I have a very good stock option, gonna be rich!"],
    
    candidate_labels=["education", "politics", "finance", "computer", "transport"],
)

[{'sequence': 'This is the Data Mining course from CPGEI',
  'labels': ['computer', 'education', 'transport', 'finance', 'politics'],
  'scores': [0.5107241272926331,
   0.33712857961654663,
   0.05968839302659035,
   0.05526594817638397,
   0.03719286620616913]},
 {'sequence': 'I have a very good stock option, gonna be rich!',
  'labels': ['finance', 'transport', 'computer', 'education', 'politics'],
  'scores': [0.8379274010658264,
   0.05080162733793259,
   0.04393560066819191,
   0.04384589567780495,
   0.023489370942115784]}]

### NER - Named Entity Recognition

In [38]:
from transformers import pipeline


#usando o modelo padrão 'dbmdz/bert-large-cased-finetuned-conll03-english' (podemos especificar outros)
ner = pipeline("ner", grouped_entities="simple")


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english)


In [39]:
ner("I am Thiago and a teach Data Mining at CPGEI.")

[{'entity_group': 'PER',
  'score': 0.9912441,
  'word': 'Thiago',
  'start': 5,
  'end': 11},
 {'entity_group': 'ORG',
  'score': 0.9959862,
  'word': 'CPGEI',
  'start': 39,
  'end': 44}]

In [40]:
## examples of transformers: https://huggingface.co/models

In [41]:
## spacy transformers: https://spacy.io/usage/embeddings-transformers#transformers